In [1]:
import torch
from transformers import AutoProcessor, LlavaForConditionalGeneration

model_id = "llava-hf/llava-1.5-7b-hf"

processor = AutoProcessor.from_pretrained(model_id)
print("Processor loaded successfully!")

model = LlavaForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True
)
print("Model loaded successfully!")

/venv/vlm-llava/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Processor loaded successfully!


Loading weights: 100%|██████████| 686/686 [00:02<00:00, 237.41it/s, Materializing param=model.vision_tower.vision_model.pre_layrnorm.weight]                        


Model loaded successfully!


In [2]:
# ============ AMBER ============
AMBER_IMG_PATH = "../data/amber/image"
QUERY_GENERATIVE_PATH = "../evaluation/AMBER/data/query/query_generative.json"

In [3]:
OUTPUT_PATH = "../results/infer/amber/llava15/normal.json"
BATCH_SIZE = 16
MAX_NEW_TOKENS = 256

In [4]:
import json
from pathlib import Path
from PIL import Image
from tqdm import tqdm

processor.tokenizer.padding_side = "left"
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token


def batch_list(items, batch_size):
    for i in range(0, len(items), batch_size):
        yield items[i:i + batch_size]


def clean_output(text):
    text = text.strip()
    if "ASSISTANT:" in text:
        text = text.split("ASSISTANT:", 1)[-1].strip()
    return text


with open(QUERY_GENERATIVE_PATH, "r", encoding="utf-8") as f:
    queries = json.load(f)

queries = [q for q in queries if 1 <= int(q["id"]) <= 1004]

print(f"Loaded {len(queries)} AMBER generative samples.")
print("Example:", queries[0])

results = []

for batch in tqdm(list(batch_list(queries, BATCH_SIZE))):
    images = []
    prompts = []
    ids = []

    for item in batch:
        image_id = int(item["id"])
        image_name = item["image"]
        prompt = item["query"]

        image_path = Path(AMBER_IMG_PATH) / image_name

        if not image_path.exists():
            matches = list(Path(AMBER_IMG_PATH).rglob(image_name))
            if len(matches) == 0:
                raise FileNotFoundError(f"Cannot find {image_name} under {AMBER_IMG_PATH}")
            image_path = matches[0]

        image = Image.open(image_path).convert("RGB")
        hf_prompt = f"USER: <image>\n{prompt} ASSISTANT:"

        images.append(image)
        prompts.append(hf_prompt)
        ids.append(image_id)

    inputs = processor(
        text=prompts,
        images=images,
        return_tensors="pt",
        padding=True
    )

    input_device = next(model.parameters()).device
    inputs = {
        k: v.to(input_device) if isinstance(v, torch.Tensor) else v
        for k, v in inputs.items()
    }

    input_len = inputs["input_ids"].shape[1]

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            use_cache=True,
            pad_token_id=processor.tokenizer.eos_token_id
        )

    generated_ids = output_ids[:, input_len:]

    outputs = processor.batch_decode(
        generated_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True
    )

    for image_id, output in zip(ids, outputs):
        response = clean_output(output)

        results.append({
            "id": image_id,
            "response": response
        })

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"\nSaved {len(results)} responses to {OUTPUT_PATH}")

Loaded 1004 AMBER generative samples.
Example: {'id': 1, 'image': 'AMBER_1.jpg', 'query': 'Describe this image.'}


100%|██████████| 63/63 [11:07<00:00, 10.60s/it]


Saved 1004 responses to ../results/infer/amber/llava15/normal.json
